# 04 — NFL translation — rookie route coverage and movement

**We are now in Notebook 04.** Load the Combine feature handoff from Notebook 03, audit regular-season NFL opportunities, and measure speed gained in the first second after the snap on GO routes without pre-snap motion.

This notebook prepares outcomes; it does not test or select features based on their relationship with NFL performance. Route labels describe concepts, not film-confirmed straight/unimpeded running. Defensive contact, alignment, scheme, and initial speed can affect movement. This is observed speed gain, not an isolated athletic ability rating.

Use rookie-season regular-season plays so earlier draft classes do not receive extra seasons of exposure. The minimum-opportunity working threshold is 10 qualifying routes per player, specified before fitting relationships. Retain all player rows and counts; mark eligibility instead of silently deleting low-opportunity players. Physical speed units remain unverified, and Combine-to-game unit equivalence is not established.


In [1]:
from pathlib import Path
import hashlib
import json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "notebooks" / "04_nfl_translation.ipynb").is_file()), None)

if ROOT is None:

    raise FileNotFoundError("Open the project or its notebooks folder first.")

FEATURES = ROOT / "outputs" / "features"

OUTPUT = ROOT / "outputs" / "nfl_translation"

OUTPUT.mkdir(parents=True, exist_ok=True)

feature_manifest_path = FEATURES / "manifest.json"

if not feature_manifest_path.is_file():

    raise FileNotFoundError("Run Notebook 03 through Section 10 first.")

feature_manifest = json.loads(feature_manifest_path.read_text())

if feature_manifest.get("schema_version") != 1:

    raise ValueError("Unsupported feature handoff version")

feature_path = FEATURES / "wr_primary_features.csv"
info = feature_manifest["files"][feature_path.name]

if hashlib.sha256(feature_path.read_bytes()).hexdigest() != info["sha256"]:

    raise ValueError("Feature handoff changed. Rerun Notebook 03 Section 10.")

features = pd.read_csv(feature_path, dtype={"nfl_id": "string", "draft_year": "Int64"})

if len(features) != info["rows"] or features.duplicated("nfl_id").any():

    raise ValueError("Unexpected player count or duplicate NFL player IDs")

print("Combine feature players:", len(features))

Combine feature players: 93


## 1. Locate local competition files

Use the existing download. There is no API login. If the files are elsewhere, set `EXISTING_DATA_DIR` to that folder. The default searches only the project's `data/` folder. On Kaggle it also checks the attached competition input. No raw tracking files are committed or copied into Git.


In [2]:
EXISTING_DATA_DIR = "/Users/hunterclark/Documents/New project/nfl-big-data-bowl-2027/data/kagglehub/competitions/nfl-big-data-bowl-2027"  # Set your existing download folder if needed.

if EXISTING_DATA_DIR:
    candidates = list(Path(EXISTING_DATA_DIR).expanduser().rglob("player_play.csv"))

else:
    candidates = list((ROOT / "data").rglob("player_play.csv"))
    
    if Path("/kaggle/input").is_dir():
        candidates += list(Path("/kaggle/input").rglob("player_play.csv"))

if len(candidates) != 1:
    
    raise ValueError(f"Expected one competition player_play.csv; found {len(candidates)}. Set EXISTING_DATA_DIR.")

DATA_DIR = candidates[0].parent
KEYS = ["game_id", "play_id", "nfl_id"]

player_play = pd.read_csv(DATA_DIR / "player_play.csv", dtype="string")
games = pd.read_csv(DATA_DIR / "games.csv", dtype="string")

if player_play.duplicated(KEYS).any() or player_play[KEYS].isna().any().any():
    raise ValueError("Missing or duplicate player-play keys")

if games["game_id"].isna().any() or games.duplicated("game_id").any():
    raise ValueError("Missing or duplicate game keys")

print("Competition data:", DATA_DIR)

Competition data: /Users/hunterclark/Documents/New project/nfl-big-data-bowl-2027/data/kagglehub/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027


## 2. Audit rookie-season route opportunities by year

Require a documented `N` penalty-nullification flag; do not infer that unknown values mean no penalty. Motion must be explicitly False. The filters and opportunity counts below are saved. A season is read from games.csv, not the calendar year in a tracking timestamp. NFL position labels and actual play roles can differ.


In [3]:
context_columns = ["nfl_id", "draft_year"]
plays = player_play.merge(features[context_columns], on="nfl_id", how="inner", validate="many_to_one")
plays = plays.merge(games[["game_id", "season", "season_type"]], on="game_id", how="left", validate="many_to_one", indicator=True)

if not plays["_merge"].eq("both").all():
    raise ValueError("A player-play game was not found in games.csv")

plays = plays.drop(columns="_merge")
plays["season"] = pd.to_numeric(plays["season"], errors="raise").astype("Int64")
plays["motion_known_false"] = plays["in_motion_at_ball_snap"].str.lower().eq("false").fillna(False)

regular = plays.loc[plays["season_type"].eq("REG")].copy()

rookie = regular.loc[regular["season"].eq(regular["draft_year"])].copy()

not_nullified = rookie.loc[rookie["play_nullified_by_penalty"].eq("N")].copy()

go_routes = not_nullified.loc[not_nullified["route_ran"].eq("GO")].copy()

candidate_plays = go_routes.loc[go_routes["motion_known_false"]].copy()

coverage = features[["nfl_id", "display_name", "draft_year"]].copy()

for name, table in [("all_player_play_rows", plays), ("regular_season_rows", regular),
                    ("rookie_regular_rows", rookie), ("rookie_non_nullified_rows", not_nullified),
                    ("rookie_go_routes", go_routes), ("candidate_go_routes", candidate_plays)]:
    coverage[name] = coverage["nfl_id"].map(table.groupby("nfl_id").size()).fillna(0).astype(int)

route_coverage_by_year = coverage.groupby("draft_year").agg(
    feature_players=("nfl_id", "size"),
    players_with_rookie_rows=("rookie_regular_rows", lambda s: int(s.gt(0).sum())),
    players_with_candidate_routes=("candidate_go_routes", lambda s: int(s.gt(0).sum())),
    candidate_routes=("candidate_go_routes", "sum"),
).reset_index()

display(route_coverage_by_year)
print("Rookie route labels before GO selection:")
display(not_nullified["route_ran"].value_counts(dropna=False).rename("player-plays").to_frame())
print("GO route motion values before selection:")
display(go_routes["in_motion_at_ball_snap"].value_counts(dropna=False).to_frame("routes"))
print("Missing motion values are excluded, not assumed stationary.")

coverage.to_csv(OUTPUT / "wr_nfl_opportunity_coverage.csv", index=False)
candidate_plays.to_csv(OUTPUT / "candidate_rookie_go_routes.csv", index=False)


,draft_year,feature_players,players_with_rookie_rows,players_with_candidate_routes,candidate_routes
0,2023,32,24,23,1053
1,2024,28,26,24,992
2,2025,33,20,20,699


Rookie route labels before GO selection:


,player-plays
route_ran,
GO,2982
HITCH,2344
CROSS,1862
IN,1705
POST,1600
OUT,1438
CORNER,1196
SLANT,863
SCREEN,408


GO route motion values before selection:


,routes
in_motion_at_ball_snap,
FALSE,2744
TRUE,236
<NA>,2


Missing motion values are excluded, not assumed stationary.


## 3. Read matched game tracking in chunks

This is the slow step: the three CSVs total approximately 2 GB. Read only needed columns and retain only exact candidate player-play keys. Tracking year filenames correspond to NFL seasons; January games remain associated with their season. Progress is printed after each season. The full unfiltered tracking dataset is never loaded into memory.


In [4]:
TRACKING_COLUMNS = KEYS + ["time", "s", "event"]
CHUNK_ROWS = 250_000

matched_parts = []
scan_rows = []

for season in sorted(candidate_plays["season"].dropna().unique()):
    season = int(season)
    keys = candidate_plays.loc[candidate_plays["season"].eq(season), KEYS].drop_duplicates()
    path = DATA_DIR / f"game_tracking_{season}.csv"
    
    if not path.is_file():
        raise FileNotFoundError(path)
    
    player_ids = set(keys["nfl_id"])
    game_ids = set(keys["game_id"])
    scanned = matched = 0
    
    print(f"Reading season {season} for {len(keys)} candidate player-plays...", flush=True)
    
    for chunk in pd.read_csv(path, usecols=TRACKING_COLUMNS, dtype="string", chunksize=CHUNK_ROWS):
        scanned += len(chunk)
        selected = chunk.loc[chunk["nfl_id"].isin(player_ids) & chunk["game_id"].isin(game_ids)]
        selected = selected.merge(keys, on=KEYS, how="inner", validate="many_to_one")
    
        if not selected.empty:
            matched_parts.append(selected)
            matched += len(selected)
    scan_rows.append({"season": season, "rows_scanned": scanned, "frames_matched": matched})
    print(f"Season {season}: {scanned:,} frames scanned; {matched:,} retained.", flush=True)

if not matched_parts:
    
    raise ValueError("No candidate game tracking found")

game_frames = pd.concat(matched_parts, ignore_index=True)
game_frames["time"] = pd.to_datetime(game_frames["time"], errors="coerce")
game_frames["s"] = pd.to_numeric(game_frames["s"], errors="coerce").astype(float)
scan_summary = pd.DataFrame(scan_rows)

display(scan_summary)
print("Matched tracking event labels:")
display(game_frames["event"].value_counts(dropna=False).to_frame("frames"))
scan_summary.to_csv(OUTPUT / "tracking_scan_summary.csv", index=False)


Reading season 2023 for 1053 candidate player-plays...
Season 2023: 3,665,443 frames scanned; 81,155 retained.
Reading season 2024 for 992 candidate player-plays...
Season 2024: 7,827,944 frames scanned; 75,606 retained.
Reading season 2025 for 699 candidate player-plays...
Season 2025: 11,160,903 frames scanned; 53,030 retained.


,season,rows_scanned,frames_matched
0,2023,3665443,81155
1,2024,7827944,75606
2,2025,11160903,53030


Matched tracking event labels:


,frames
event,
<NA>,196240
ball_snap,2744
pass_forward,2441
pass_arrived,1792
pass_outcome_caught,1363
tackle,1208
first_contact,1115
pass_outcome_incomplete,957
autoevent_passinterrupted,366


## 4. Measure first-second speed gain

Working movement outcome: supplied speed at snap +1.0 second minus supplied speed at the snap, on candidate GO routes. Require exactly one snap event and one pass-forward event in this player's recording, with the pass at or after +1.0 second. Require valid nonnegative speed and complete timing coverage in that segment. Interpolate the endpoint only between observed frames at most 0.15 seconds apart; do not extrapolate. Save a status for every candidate route, including routes with no matched tracking.

No maximum acceleration, unknown physical launch, or Combine threshold alignment is required. Initial speed is retained as a necessary context variable for later models. Excluding missing events and quick throws selects a subset of routes; report this selection. A labeled GO route can still include contact or an initial release move. This outcome measures speed gained along the actual route, not pure forward acceleration.


In [5]:
HORIZON_S = 1.0
MAX_GAP_S = 0.15
MIN_ROUTES = 10

def first_second_gain(frames):
    frames = frames.sort_values("time")
    
    if frames["time"].isna().any():
        return {"tracking_status": "Invalid timestamps"}
    
    if frames["time"].duplicated().any():
        return {"tracking_status": "Duplicate timestamps"}
    
    snap = frames.loc[frames["event"].eq("ball_snap"), "time"]
    release = frames.loc[frames["event"].eq("pass_forward"), "time"]
    
    if len(snap) != 1:
        return {"tracking_status": "Missing or multiple snap events"}
    
    if len(release) != 1:
        return {"tracking_status": "Missing or multiple pass events"}
    
    throw_time = (release.iloc[0] - snap.iloc[0]).total_seconds()
    
    if throw_time < HORIZON_S - 1e-8:
        return {"tracking_status": "Pass before measurement horizon"}
    
    t = (frames["time"] - snap.iloc[0]).dt.total_seconds().to_numpy()
    speed = frames["s"].to_numpy(dtype=float)
    segment_indices = np.flatnonzero(t >= 0)
    
    if not len(segment_indices) or abs(t[segment_indices[0]]) > 1e-8 or t[-1] < HORIZON_S - 1e-8:
        return {"tracking_status": "Incomplete snap-to-horizon coverage"}
    
    start = int(segment_indices[0])
    end = int(np.searchsorted(t, HORIZON_S - 1e-8))
    
    if end >= len(t):
        return {"tracking_status": "Incomplete snap-to-horizon coverage"}
    
    segment_t = t[start:end+1]
    segment_s = speed[start:end+1]
    
    if not np.isfinite(segment_s).all() or (segment_s < 0).any():
        return {"tracking_status": "Invalid speed in measurement segment"}
    
    if (np.diff(segment_t) > MAX_GAP_S + 1e-8).any():
        return {"tracking_status": "Gap in measurement segment"}
    
    speed_end = float(np.interp(HORIZON_S, segment_t, segment_s))
    
    return {"tracking_status": "Measured", "snap_speed_raw": float(segment_s[0]),
            "speed_at_one_second_raw": speed_end,
            "first_second_speed_gain_raw": speed_end-float(segment_s[0]),
            "snap_to_pass_s": throw_time}

movement_rows = []

for key, frames in game_frames.groupby(KEYS, dropna=False, sort=False):
    movement_rows.append({**dict(zip(KEYS, key)), **first_second_gain(frames)})

movement = pd.DataFrame(movement_rows)
play_outcomes = candidate_plays.merge(movement, on=KEYS, how="left", validate="one_to_one")

play_outcomes["tracking_status"] = play_outcomes["tracking_status"].fillna("No matched tracking")

print("Every candidate route, including failures:")
display(pd.crosstab(play_outcomes["draft_year"], play_outcomes["tracking_status"]))

valid_plays = play_outcomes.loc[play_outcomes["tracking_status"].eq("Measured")].copy()

if valid_plays.empty:
    raise ValueError("No route outcomes measured; inspect event and status coverage before proceeding")

print("Observed movement summary (raw units):")
display(valid_plays[["snap_speed_raw", "speed_at_one_second_raw", "first_second_speed_gain_raw", "snap_to_pass_s"]].describe().round(4))
play_outcomes.to_csv(OUTPUT / "rookie_go_route_outcomes.csv", index=False)


Every candidate route, including failures:


tracking_status,Measured,Missing or multiple pass events
draft_year,,
2023,938,115
2024,875,117
2025,628,71


Observed movement summary (raw units):


,snap_speed_raw,speed_at_one_second_raw,first_second_speed_gain_raw,snap_to_pass_s
count,2441.0000,2441.0000,2441.0000,2441.0000
mean,0.0921,4.0731,3.9810,2.8558
std,0.1700,0.8722,0.8327,1.0711
min,0.0000,0.0000,0.0000,1.0000
25%,0.0000,3.4900,3.4200,2.2000
50%,0.0300,4.1600,4.0800,2.6000
75%,0.1100,4.7000,4.6000,3.2000
max,1.6900,6.6500,6.4300,12.7000


## 5. Save player-level outcome coverage for Notebook 05

Retain all 93 Combine feature players, including those with no usable NFL routes. The working player outcome is mean first-second speed gain across measured candidate routes. Also retain medians, route counts, snap-speed context, throw time, and exposure variability. A mean pools different coverage/release contexts; Notebook 05 must account for available context and show sensitivity instead of calling this an isolated athletic trait.

The main eligibility indicator requires at least 10 measured routes and at least two measured Combine intervals. Keep single-recording players and low-opportunity players in the saved audit for sensitivity analyses. Do not impute absent NFL outcomes. The 10-route threshold is a working precision/opportunity rule, not a claim of proven reliability. No Combine-to-NFL association is fitted in this notebook.


In [6]:
for column in ["separation_at_pass_forward", "cushion", "yards_to_go"]:
    valid_plays[column] = pd.to_numeric(valid_plays[column], errors="coerce")

player_movement = valid_plays.groupby("nfl_id").agg(
    measured_go_routes=("play_id", "size"), measured_games=("game_id", "nunique"),
    mean_first_second_speed_gain_raw=("first_second_speed_gain_raw", "mean"),
    median_first_second_speed_gain_raw=("first_second_speed_gain_raw", "median"),
    std_first_second_speed_gain_raw=("first_second_speed_gain_raw", "std"),
    mean_snap_speed_raw=("snap_speed_raw", "mean"), mean_snap_to_pass_s=("snap_to_pass_s", "mean"),
    mean_separation_at_throw_raw=("separation_at_pass_forward", "mean"),
    routes_with_separation=("separation_at_pass_forward", "count"),
).reset_index()

player_outcomes = features.merge(player_movement, on="nfl_id", how="left", validate="one_to_one")
player_outcomes = player_outcomes.merge(
    coverage[["nfl_id", "candidate_go_routes"]], on="nfl_id", how="left", validate="one_to_one"
)

for column in ["measured_go_routes", "measured_games", "routes_with_separation"]:
    player_outcomes[column] = player_outcomes[column].fillna(0).astype(int)
player_outcomes["meets_nfl_opportunity_rule"] = player_outcomes["measured_go_routes"].ge(MIN_ROUTES)
player_outcomes["main_analysis_eligible"] = (
    player_outcomes["meets_nfl_opportunity_rule"] & player_outcomes["measured_attempts"].ge(2)
)

final_coverage = player_outcomes.groupby("draft_year").agg(
    feature_players=("nfl_id", "size"),
    players_with_measured_routes=("measured_go_routes", lambda s:int(s.gt(0).sum())),
    measured_routes=("measured_go_routes", "sum"),
    players_with_10_routes=("meets_nfl_opportunity_rule", "sum"),
    main_analysis_players=("main_analysis_eligible", "sum"),
).reset_index()

display(final_coverage)
print("Coverage by defensive scheme among measured plays:")
display(pd.crosstab(valid_plays["draft_year"], valid_plays["team_coverage_man_zone"].fillna("Unknown")))

nfl_decisions = {
    "season_window": "Rookie NFL season only, regular season, documented non-nullified plays",
    "route_population": "GO routes with explicit False pre-snap motion flag; not film-verified unimpeded releases",
    "movement_outcome": "Mean speed gained from snap to +1.0 s; pass must occur at or after +1.0 s",
    "time_coverage": "Require snap/pass events; segment speed valid; max gap 0.15 s; bounded interpolation only",
    "eligibility": "Working main rule: >=10 measured routes and >=2 measured Combine intervals",
    "initial_speed": "Retained; must be considered in Notebook 05 alongside release/coverage context",
    "secondary_separation": "Available summary only; throw timing and coverage affect it; no association tested",
    "year": "Audit counts by draft year; year required in both later models and within-year sensitivity",
    "selection": "Eligibility selects players with opportunity and observable non-motion GO routes; do not generalize to all prospects",
    "units": "Raw tracking speed units; physical units and cross-source equivalence unverified",
    "association": "None fitted; no NFL relationship or predictive improvement established",
}

exports = {
    "wr_nfl_player_outcomes.csv": player_outcomes,
    "wr_nfl_outcome_coverage_by_year.csv": final_coverage,
}

nfl_manifest = {"schema_version":1,"created_utc":datetime.now(timezone.utc).isoformat(),
                "source_feature_manifest_sha256":hashlib.sha256(feature_manifest_path.read_bytes()).hexdigest(),
                "decisions":nfl_decisions,"files":{}}

for filename, table in exports.items():
    target = OUTPUT / filename
    table.to_csv(target,index=False)
    nfl_manifest["files"][filename] = {"rows":len(table),"sha256":hashlib.sha256(target.read_bytes()).hexdigest()}

route_path = OUTPUT / "rookie_go_route_outcomes.csv"

nfl_manifest["files"][route_path.name] = {"rows":len(play_outcomes),"sha256":hashlib.sha256(route_path.read_bytes()).hexdigest()}
(OUTPUT / "manifest.json").write_text(json.dumps(nfl_manifest,indent=2)+"\n")

display(pd.Series(nfl_decisions,name="Decision").to_frame())
print("Notebook 04 outcomes saved:",OUTPUT)
print("No model has been fitted. Review coverage before starting Notebook 05.")


,draft_year,feature_players,players_with_measured_routes,measured_routes,players_with_10_routes,main_analysis_players
0,2023,32,23,938,19,17
1,2024,28,24,875,19,12
2,2025,33,20,628,15,14


Coverage by defensive scheme among measured plays:


team_coverage_man_zone,MAN_COVERAGE,Unknown,ZONE_COVERAGE
draft_year,,,
2023,250,1,687
2024,273,0,602
2025,185,0,443


,Decision
season_window,"Rookie NFL season only, regular season, docume..."
route_population,GO routes with explicit False pre-snap motion ...
movement_outcome,Mean speed gained from snap to +1.0 s; pass mu...
time_coverage,Require snap/pass events; segment speed valid;...
eligibility,Working main rule: >=10 measured routes and >=...
initial_speed,Retained; must be considered in Notebook 05 al...
secondary_separation,Available summary only; throw timing and cover...
year,Audit counts by draft year; year required in b...
selection,Eligibility selects players with opportunity a...
units,Raw tracking speed units; physical units and c...


Notebook 04 outcomes saved: /Users/hunterclark/Documents/New project/nfl-big-data-bowl-2027/outputs/nfl_translation
No model has been fitted. Review coverage before starting Notebook 05.
